# The Ninefold Tide — Targeted Child/Teen Original Voice Retakes

Correct the two weak reference voices **Elen (14)** and **Mari (6)** without redoing other cast. Generates four original candidates for each. All audition files stay unapproved pending evaluation. Caches pretrained weights only in Kaggle temporary disk, not phone/mobile data.


In [ ]:
from pathlib import Path
import json,os,sys,subprocess,zipfile,time
from datetime import datetime,timezone
os.environ["HF_HOME"]="/kaggle/temp/ninefold_young_huggingface_cache"
os.environ["HF_HUB_DISABLE_TELEMETRY"]="1"
root=Path("/kaggle/temp/ninefold_young_cast_source")
out=Path("/kaggle/working/ninefold_young_voice_retakes")
out.mkdir(parents=True,exist_ok=True)
status={"project":"The Ninefold Tide", "scope":"Elen and Mari voice retakes, 4 original options each",
 "approval":"PROVISIONAL_UNAPPROVED", "state":"STARTED",
 "started_utc":datetime.now(timezone.utc).isoformat(), "completed_steps":[]}
def save():
    (out/"batch_status.json").write_text(json.dumps(status,indent=2)+"\n",encoding="utf8")
def step(name,argv,timeout):
    t=time.monotonic()
    print("START",name,flush=True)
    p=subprocess.run(argv,text=True,capture_output=True,timeout=timeout)
    (out/("log_"+name.replace(" ","_")+".txt")).write_text(
        (p.stdout or "")[-18000:]+"\n--- STDERR ---\n"+(p.stderr or "")[-18000:],encoding="utf8")
    print("END",name,"return",p.returncode,"elapsed",round(time.monotonic()-t,1),flush=True)
    if p.returncode:
        raise RuntimeError(name+": "+(p.stderr or p.stdout or "")[-450:])
    status["completed_steps"].append(name)
    save()
save()
try:
    step("Check NVIDIA CUDA",["nvidia-smi","--query-gpu=name,memory.total,memory.free","--format=csv,noheader"],40)
    step("Install Qwen TTS", [sys.executable,"-m","pip","install","-q","qwen-tts","soundfile"],500)
    step("Clone exact series production source",
         ["git","clone","-q","--depth","1","https://github.com/eghosa001/TOOLS.git",str(root)],130)
    worker=root/"video/ninefold_tide/audio/young_voice_retakes.py"
    step("Generate eight original youthful voice candidates",
         [sys.executable,str(worker),"--output",str(out),"--allow-download"],1600)
    result=json.loads((out/"young_voice_retakes_manifest.json").read_text(encoding="utf8"))
    status["generated_voice_candidates"]=result["count"]
    status["state"]="AUDITION_REFERENCES_GENERATED_UNAPPROVED" if result["count"]==8 else "INCOMPLETE"
except Exception as exc:
    status["state"]="PARTIAL_OR_FAILED"
    status["error"]=str(exc)[:1100]
    print("AUDITION_RUN_BLOCKED",str(exc)[:450],flush=True)
finally:
    status["ended_utc"]=datetime.now(timezone.utc).isoformat()
    save()
    archive=Path("/kaggle/working/ninefold_elen_mari_retakes_UNAPPROVED.zip")
    with zipfile.ZipFile(archive,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=5) as z:
        for path in out.rglob("*"):
            if path.is_file() and path.suffix.lower() in (".wav",".json",".txt"):
                z.write(path,arcname=path.relative_to(out))
    print("AUDITION_ARCHIVE",str(archive),archive.stat().st_size,status["state"],flush=True)
    print("All new Elen and Mari references remain unapproved until compared and youth voice quality verified.",flush=True)
